In [1]:
# """
# Train a Vision Transformer (ViT) from scratch (randomly initialized, NOT
# pretrained) on Imagenette, where image paths and labels come from a CSV
# file in /Data — and sweep over ViT hyperparameters (patch size, embedding
# dimension, attention heads, hidden layers), changing only ONE parameter
# at a time relative to a fixed baseline configuration.
 
# For each combination tried, this script:
#   - trains a fresh ViT from scratch for EPOCHS epochs
#   - saves an accuracy-over-time plot for that combination, with a title
#     describing exactly which hyperparameters were used
#   - records the best (maximum) validation accuracy reached
 
# At the end, it writes a summary CSV with one row per combination, so you
# can compare all the max validation accuracies side by side.
 
# Needs: torch, torchvision, matplotlib, tqdm (no Hugging Face libraries).


In [2]:
import csv
import gc
import os
import random
 
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
%pwd

'/home/geoff/Uni'

In [4]:
# ---------------------------------------------------------------------------
# 0. CONFIG — adjust these to match your CSV
# ---------------------------------------------------------------------------

DATA_DIR = "./imagenette2/imagenette2"
CSV_PATH = "./imagenette2/imagenette2/noisy_imagenette.csv"   # <-- change to your actual CSV filename
HAS_HEADER = True
VAL_FRACTION = 0.2              # 20% of rows held out for validation
RANDOM_SEED = 42

In [5]:
# ---------------------------------------------------------------------------
# 1. READ THE CSV AND BUILD (path, label) PAIRS
# ---------------------------------------------------------------------------
def load_csv_rows(csv_path, has_header):
#    """Read the CSV and return a list of (image_path, label_str) tuples."""
    rows = []
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        if has_header:
            next(reader)  # skip header row
        for row in reader:
            image_path, label = row[0], row[1]
            rows.append((image_path, label))
    return rows


all_rows = load_csv_rows(CSV_PATH, HAS_HEADER)

# Build a sorted list of unique class names, then a name -> integer index
# mapping (models need integer labels, not strings).
class_names = sorted({label for _, label in all_rows})
label_to_index = {name: i for i, name in enumerate(class_names)}
num_labels = len(class_names)
print(f"Found {num_labels} classes: {class_names}")

# Shuffle deterministically, then split into train/val since the CSV
# itself doesn't mark a split.
random.seed(RANDOM_SEED)
random.shuffle(all_rows)

split_index = int(len(all_rows) * (1 - VAL_FRACTION))
train_rows = all_rows[:split_index]
val_rows = all_rows[split_index:]
print(f"Train rows: {len(train_rows)} | Val rows: {len(val_rows)}")




Found 10 classes: ['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']
Train rows: 10715 | Val rows: 2679


In [6]:
# ---------------------------------------------------------------------------
# 2. A CUSTOM DATASET THAT READS IMAGES USING THE CSV ROWS
# ---------------------------------------------------------------------------
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),                 # PIL image -> [0,1] tensor, shape (C,H,W)
    transforms.Normalize(mean=MEAN, std=STD),
])


class CsvImageDataset(Dataset):
#    """
#    A minimal PyTorch Dataset that, for each CSV row, loads the image at
#    `data_dir/path` from disk and returns (transformed_image, label_index).
#    """

    def __init__(self, rows, data_dir, label_to_index, transform):
        self.rows = rows
        self.data_dir = data_dir
        self.label_to_index = label_to_index
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        image_path, label_str = self.rows[idx]
        full_path = os.path.join(self.data_dir, image_path)

        image = Image.open(full_path).convert("RGB")  # force 3 channels
        image = self.transform(image)

        label = self.label_to_index[label_str]
        return image, label


train_data = CsvImageDataset(train_rows, DATA_DIR, label_to_index, transform)
val_data = CsvImageDataset(val_rows, DATA_DIR, label_to_index, transform)

# These dataloaders are shared by every hyperparameter combination below —
# the data itself doesn't change, only the model architecture and training
# hyperparameters do.
BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.enabled = False


In [7]:
# ---------------------------------------------------------------------------
# 3. VISION TRANSFORMER MODEL DEFINITION
# ---------------------------------------------------------------------------
class PatchEmbedding(nn.Module):
#    """
#    Splits an image into non-overlapping patch_size x patch_size patches
#    and linearly projects each one into an embed_dim-length vector, via a
#    Conv2d whose kernel size AND stride both equal patch_size (so each
#    "convolution" touches exactly one non-overlapping patch).
#    """
 
    def __init__(self, patch_size, num_channels, embed_dim):
        super().__init__()
        self.projection = nn.Conv2d(
            num_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )
 
    def forward(self, x):
        x = self.projection(x)   # (batch, embed_dim, H/patch, W/patch)
        x = x.flatten(2)         # (batch, embed_dim, num_patches)
        x = x.transpose(1, 2)    # (batch, num_patches, embed_dim)
        return x
 
 
class VisionTransformer(nn.Module):
    def __init__(
        self,
        image_size,
        patch_size,
        num_channels,
        embed_dim,
        num_heads,
        num_layers,
        num_classes,
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()
        num_patches = (image_size // patch_size) ** 2
 
        self.patch_embed = PatchEmbedding(patch_size, num_channels, embed_dim)
 
        # A learnable "classification token" prepended to the patch
        # sequence; after self-attention mixes information between all
        # patches, this token's final embedding represents the whole image.
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
 
        # Learnable position embeddings — one per patch, plus one for the
        # [CLS] token — since self-attention has no built-in notion of
        # where each patch sits in the image.
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)
 
        # `norm_first=True` applies LayerNorm BEFORE attention / feed-forward
        # in each block ("pre-norm") instead of after. Pre-norm is what most
        # ViT implementations use because post-norm (PyTorch's default)
        # becomes unstable as models get wider/deeper, which showed up as
        # flat, non-learning training at embed_dim=768.
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, enable_nested_tensor=False
        )
 
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
 
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
 
    def forward_features(self, x):
        batch_size = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)
        x = self.encoder(x)
        x = self.norm(x)
        return x[:, 0]
 
    def forward(self, x):
        cls_embedding = self.forward_features(x)
        return self.head(cls_embedding)
 

In [8]:
# ---------------------------------------------------------------------------
# 4. TRAIN-ONE-CONFIGURATION FUNCTION
# ---------------------------------------------------------------------------
# Everything needed to build, train, and evaluate a ViT for ONE particular
# choice of (patch_size, embed_dim, num_heads, num_layers) is wrapped in
# this function, so the hyperparameter sweep in step 5 can just call it
# once per combination.
EPOCHS = 15
LEARNING_RATE = 1e-4    # lowered from 3e-4: wide models (embed_dim=768) failed to learn at the higher rate
WARMUP_EPOCHS = 5       # lengthened from 2: wide models are most sensitive to the earliest steps
WEIGHT_DECAY = 0.05
MAX_GRAD_NORM = 1.0
MLP_RATIO = 4.0
DROPOUT = 0.1
 
 
def train_one_configuration(patch_size, embed_dim, num_heads, num_layers, title, filename_tag):
#    """
#    Builds a fresh, randomly-initialized ViT with the given hyperparameters,
#    trains it from scratch for EPOCHS epochs, saves an accuracy-over-time
#    plot titled `title`, and returns the maximum validation accuracy reached.
#    """
    assert IMAGE_SIZE % patch_size == 0, "IMAGE_SIZE must be divisible by patch_size"
    assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
 
    model = VisionTransformer(
        image_size=IMAGE_SIZE,
        patch_size=patch_size,
        num_channels=3,
        embed_dim=embed_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        num_classes=num_labels,
        mlp_ratio=MLP_RATIO,
        dropout=DROPOUT,
    ).to(device)
 
    num_params = sum(p.numel() for p in model.parameters())
    print(f"\n=== {title} ({num_params:,} parameters) ===")
 
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
 
    # Linear warmup + cosine decay, same rationale as before: jumping
    # straight to the target learning rate destabilizes Transformers early
    # in training, so ramp up first, then ease back down.
    steps_per_epoch = len(train_loader)
    warmup_steps = WARMUP_EPOCHS * steps_per_epoch
    total_steps = EPOCHS * steps_per_epoch
 
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return current_step / max(1, warmup_steps)
        progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + torch.cos(torch.tensor(progress * 3.14159265)).item())
 
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
 
    train_accuracies = []
    val_accuracies = []
 
    def run_validation():
        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                logits = model(images)
                loss = loss_fn(logits, targets)
                total_loss += loss.item() * images.size(0)
                predictions = logits.argmax(dim=1)
                correct += (predictions == targets).sum().item()
                total += targets.size(0)
        return total_loss / total, correct / total
 
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
 
        progress_bar = tqdm(
            train_loader, desc=f"[{filename_tag}] Epoch {epoch + 1}/{EPOCHS}", unit="batch"
        )
 
        samples_seen = 0
        for images, targets in progress_bar:
            images, targets = images.to(device), targets.to(device)
 
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, targets)
            loss.backward()
 
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
 
            optimizer.step()
            scheduler.step()
 
            running_loss += loss.item() * images.size(0)
            samples_seen += images.size(0)
 
            predictions = logits.argmax(dim=1)
            correct_train += (predictions == targets).sum().item()
            total_train += targets.size(0)
 
            progress_bar.set_postfix(
                loss=running_loss / samples_seen,
                lr=optimizer.param_groups[0]["lr"],
            )
 
        train_loss = running_loss / len(train_data)
        train_acc = correct_train / total_train
        val_loss, val_acc = run_validation()
 
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)
 
        print(
            f"[{filename_tag}] Epoch {epoch + 1}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )
 
    # --- Save the accuracy-over-time plot for this combination. ---
    epochs_range = range(1, EPOCHS + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, train_accuracies, marker="o", label="Train accuracy")
    plt.plot(epochs_range, val_accuracies, marker="o", label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True)
 
    plot_path = f"./accuracy_{filename_tag}.png"
    plt.savefig(plot_path)
    plt.close()  # close (rather than show) so the sweep doesn't pause for each plot
    print(f"Saved accuracy plot to {plot_path}")
 
    max_val_acc = max(val_accuracies)
 
    # --- Free this run's GPU memory before the next combination starts. ---
    # Without this, the model/optimizer/scheduler from THIS run stay
    # resident on the GPU while the next one is built, and over enough
    # combinations that steadily accumulates until CUDA runs out of
    # memory. Deleting the references and clearing PyTorch's cache
    # reclaims that memory for the next run.
    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
 
    return max_val_acc
 
 


In [9]:
# ---------------------------------------------------------------------------
# 5. HYPERPARAMETER SWEEP — FULL GRID
# ---------------------------------------------------------------------------
# Every combination of every value below is tried: 3 * 3 * 3 * 3 = 81 full
# training runs in total. This is exhaustive (it directly compares every
# possible combination, not just single-parameter effects around one
# baseline), but is also 81x the compute of training just one
# configuration — expect this to take a long time. Lower EPOCHS in step 4
# for the sweep itself if you want a faster first pass.
import itertools
 
PARAM_GRIDS = {
    "patch_size": [8, 16, 32],
    "embed_dim": [192, 384, 768],
    "num_heads": [3, 6, 12],
    "num_layers": [3, 6, 12],
}

# Start the sweep at this combination number (1-based). Combinations before
# it are skipped. Set to 1 to run from the beginning.
START_AT = 21

# `itertools.product` generates every combination of the 4 lists above,
# i.e. the full Cartesian product / grid.
combinations_to_run = [
    dict(zip(PARAM_GRIDS.keys(), values))
    for values in itertools.product(*PARAM_GRIDS.values())
]
 
# ---------------------------------------------------------------------------
# QUICK TEST SWITCH
# ---------------------------------------------------------------------------
# Set QUICK_TEST = True to run ONLY the single combination that failed to
# learn before (embed_dim=768), for a short run, to check whether the new
# learning rate / warmup / pre-norm settings fix it BEFORE committing to
# all 81 runs. Set it back to False for the full sweep.
#
# 10 epochs is used rather than fewer because the first 5 are warmup —
# a shorter run would never leave the warmup phase.
QUICK_TEST = False
QUICK_TEST_COMBO = {"patch_size": 16, "embed_dim": 768, "num_heads": 12, "num_layers": 6}
QUICK_TEST_EPOCHS = 10
 
if QUICK_TEST:
    combinations_to_run = [QUICK_TEST_COMBO]
    EPOCHS = QUICK_TEST_EPOCHS   # train_one_configuration reads this global
    print(f"QUICK_TEST is ON: running only {QUICK_TEST_COMBO} for {EPOCHS} epochs")
 
print(f"\nRunning {len(combinations_to_run)} combinations (full grid sweep)...")
 
# Quick-test results go to a separate file so they never mix with the real sweep.
summary_path = "./quick_test_results.csv" if QUICK_TEST else "./hyperparameter_sweep_results.csv"
fieldnames = ["patch_size", "embed_dim", "num_heads", "num_layers", "max_val_accuracy"]
 
# Write the header now, then append one row per combination as soon as it
# finishes (see the loop below) — rather than writing the whole file only
# once every combination is done. This way, if the sweep crashes partway
# through (e.g. an OOM later in the run), every combination that already
# finished is safely on disk instead of lost.
with open(summary_path, "w", newline="") as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()
 
results = []
for i, combo in enumerate(combinations_to_run, start=1):
    patch_size = combo["patch_size"]
    embed_dim = combo["embed_dim"]
    num_heads = combo["num_heads"]
    num_layers = combo["num_layers"]
 
    # Skip combinations that aren't architecturally valid: each attention
    # head needs an equal, whole-number share of the embedding dimension.
    if embed_dim % num_heads != 0:
        print(
            f"Skipping combo {i}/{len(combinations_to_run)}: "
            f"embed_dim={embed_dim} not divisible by num_heads={num_heads}"
        )
        continue
    if i < START_AT:
        continue
    
    filename_tag = f"p{patch_size}_e{embed_dim}_h{num_heads}_l{num_layers}"
    title = (
        f"Accuracy over time ({i}/{len(combinations_to_run)})\n"
        f"patch_size={patch_size}, embed_dim={embed_dim}, "
        f"num_heads={num_heads}, num_layers={num_layers}"
    )
 
    try:
        max_val_acc = train_one_configuration(
            patch_size=patch_size,
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_layers=num_layers,
            title=title,
            filename_tag=filename_tag,
        )
    except torch.cuda.OutOfMemoryError:
        # Even with cleanup between runs, a particularly large combination
        # (big embed_dim/num_layers) can still be too much to fit. Log it
        # and move on rather than losing every result gathered so far.
        print(f"Skipping combo {i}/{len(combinations_to_run)} ({filename_tag}): CUDA out of memory")
        gc.collect()
        torch.cuda.empty_cache()
        continue
 
    result_row = {
        "patch_size": patch_size,
        "embed_dim": embed_dim,
        "num_heads": num_heads,
        "num_layers": num_layers,
        "max_val_accuracy": max_val_acc,
    }
    results.append(result_row)
 
    # Append immediately, so this result is saved even if a later
    # combination crashes the sweep.
    with open(summary_path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writerow(result_row)
 


Running 81 combinations (full grid sweep)...

=== Accuracy over time (21/81)
patch_size=8, embed_dim=768, num_heads=3, num_layers=12 (85,815,562 parameters) ===


[p8_e768_h3_l12] Epoch 1/15:   1%|▎                        | 4/335 [03:29<4:48:38, 52.32s/batch, loss=2.37, lr=2.39e-7]


KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------------------------
# 6. SUMMARY
# ---------------------------------------------------------------------------
# Every row was already appended to summary_path as its combination
# finished (see step 5), so there's nothing left to write here — just
# report what was saved.
print(f"\nSaved {len(results)} results to {summary_path}")
for row in results:
    print(row)

In [ ]:
import sys, gc, torch
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()
torch.cuda.empty_cache()

In [ ]:
nvidia-smi
